# 04 · Triton 并行划分:grid、program_id、tl.arange 与 mask

> Learn Triton 系列 · 阶段 1(编程模型)第 1 篇
> 前置:第 03 篇(第一个 kernel 的骨架)
> 运行环境:Google Colab T4 GPU

第 03 篇的 `add_kernel` 已经用到了 `grid`、`tl.program_id`、`tl.arange`、`mask` 四件套,但只是照着模板写。本篇把这四件套彻底讲透:**工作量是怎么切给成百上千个 program 的、二维问题怎么切、一个 program 怎么处理多块数据、mask 有哪些用法**。这是后面所有 kernel(GEMM、FlashAttention)的地基语法。

## 环境准备

In [ ]:
import torch

assert torch.cuda.is_available(), "请在 Colab 选择 GPU 运行时"

import triton
import triton.language as tl

print(f"PyTorch {torch.__version__} | Triton {triton.__version__} | {torch.cuda.get_device_name(0)}")


def gpu_time_ms(fn, warmup=5, repeat=20):
    for _ in range(warmup):
        fn()
    torch.cuda.synchronize()
    times = []
    for _ in range(repeat):
        s, e = torch.cuda.Event(enable_timing=True), torch.cuda.Event(enable_timing=True)
        s.record(); fn(); e.record()
        torch.cuda.synchronize()
        times.append(s.elapsed_time(e))
    times.sort()
    return times[len(times) // 2]

---

## §1 是什么 & 能力边界

### grid:把工作量切成多少份

启动 kernel 时的 `kernel[grid](...)` 中,`grid` 是一个 1~3 维的元组,声明**启动多少个 program 实例**。每个实例用 `tl.program_id(axis=0/1/2)` 得知自己的坐标,据此推算自己负责哪块数据。

- `grid=(N,)`:一维划分,适合向量(第 03 篇);
- `grid=(M, N)`:二维划分,适合矩阵——比如 GEMM 里 `pid_m` 管第几行块、`pid_n` 管第几列块;
- `grid` 可以是 `lambda meta: ...`:在 autotune 改变 BLOCK_SIZE 时,grid 自动跟着变。

### tl.arange 与块内索引

`tl.arange(0, BLOCK)` 生成 `[0,1,...,BLOCK-1]` 的**编译期定长向量**,是构造块内偏移的唯一方式。两条硬规则:

1. 两个参数都必须是编译期常量(`tl.constexpr` 或字面量);
2. **长度必须是 2 的幂**(否则编译报错)——所以任意长度的数据都要靠"向上取整的 grid + mask"覆盖。

### mask:边界安全网,也是逻辑工具

`mask` 不只防越界,它是 Triton 里表达"条件读写"的通用机制:

- `tl.load(ptr + offs, mask=m, other=0.0)`:无效位置不发起访存,用 `other` 填充——归约时填充值的选择(加法填 0、求 max 填 -inf)直接影响正确性;
- `tl.store(ptr + offs, val, mask=m)`:无效位置不写;
- 后面会见到的用法:causal attention 的下三角 mask(第 17 篇)、paged KV 的有效长度 mask(第 18 篇),本质都是它。

### 能做什么 / 不能做什么

能做:

- 任意维度、任意非整除长度的数据划分(grid 向上取整 + mask);
- 一个 program 既可以管一块数据,也可以循环处理多块(例 3 的 grid-stride 模式);
- 通过 grid 形状与数据块的映射关系,控制访存局部性(第 11 篇 L2 swizzle 的伏笔)。

不能做:

- program 之间**不能直接通信或同步**:Triton 没有跨 program 的 barrier,需要协作时只能拆成多个 kernel 或用 atomic(第 13 篇)——这是与 CUDA(块内 `__syncthreads`)层级不同的限制,Triton 的"块内同步"由编译器隐式处理;
- grid 一旦启动就固定,kernel 内不能动态增减 program;
- `tl.arange` 不能用运行期变量做长度;变长数据(如变长序列)的处理模式是"按最大块切 + mask",而不是动态形状。

---

## §2 递进式例子

### 例 1:二维 grid —— 给矩阵加法分块

把 `M×N` 矩阵切成 `BLOCK_M×BLOCK_N` 的瓦片(tile),每个 program 负责一片。这是 GEMM(第 10 篇)的索引骨架,先在最简单的加法上练熟。

In [ ]:
@triton.jit
def add2d_kernel(
    x_ptr, y_ptr, out_ptr,
    M, N,                      # 矩阵形状(运行期)
    stride_m, stride_n,        # 行/列方向的内存步长(下一篇详讲,行优先时 = (N, 1))
    BLOCK_M: tl.constexpr, BLOCK_N: tl.constexpr,
):
    pid_m = tl.program_id(axis=0)   # 我管第几个行块
    pid_n = tl.program_id(axis=1)   # 我管第几个列块

    # 本块覆盖的行号和列号(各是一个向量)
    rows = pid_m * BLOCK_M + tl.arange(0, BLOCK_M)        # [BLOCK_M]
    cols = pid_n * BLOCK_N + tl.arange(0, BLOCK_N)        # [BLOCK_N]

    # 广播成二维:offs[i,j] = rows[i]*stride_m + cols[j]*stride_n
    offs = rows[:, None] * stride_m + cols[None, :] * stride_n   # [BLOCK_M, BLOCK_N]
    mask = (rows[:, None] < M) & (cols[None, :] < N)             # 行列各自防越界

    x = tl.load(x_ptr + offs, mask=mask)
    y = tl.load(y_ptr + offs, mask=mask)
    tl.store(out_ptr + offs, x + y, mask=mask)


def triton_add2d(x, y):
    M, N = x.shape
    out = torch.empty_like(x)
    BLOCK_M, BLOCK_N = 32, 64
    grid = (triton.cdiv(M, BLOCK_M), triton.cdiv(N, BLOCK_N))
    add2d_kernel[grid](x, y, out, M, N, x.stride(0), x.stride(1),
                       BLOCK_M=BLOCK_M, BLOCK_N=BLOCK_N)
    return out


x = torch.randn(1000, 777, device="cuda")  # 两个维度都不整除,考验 mask
y = torch.randn(1000, 777, device="cuda")
torch.testing.assert_close(triton_add2d(x, y), x + y)
print("二维分块加法正确性通过 (1000x777, BLOCK 32x64, grid =",
      (triton.cdiv(1000, 32), triton.cdiv(777, 64)), ")")

注意 `rows[:, None]` / `cols[None, :]`:Triton 支持 NumPy 风格的广播,一维向量外积式地组合成二维偏移矩阵——这一行是所有二维 kernel 的核心 idiom,务必看懂。

### 例 2:mask 的 `other` 参数 —— 填充值决定归约正确性

In [ ]:
@triton.jit
def row_max_kernel(x_ptr, out_ptr, N, BLOCK_N: tl.constexpr, BAD_FILL: tl.constexpr):
    """每个 program 求一行的最大值。BAD_FILL 用来演示错误填充值的后果。"""
    row = tl.program_id(0)
    cols = tl.arange(0, BLOCK_N)
    mask = cols < N
    fill = float("-inf") if not BAD_FILL else 0.0
    vals = tl.load(x_ptr + row * N + cols, mask=mask, other=fill)
    tl.store(out_ptr + row, tl.max(vals, axis=0))


x = torch.randn(8, 100, device="cuda") - 5.0  # 所有元素都是负数!
BLOCK_N = triton.next_power_of_2(100)          # 128:必须 2 的幂,越界部分靠 other 填充

out_good = torch.empty(8, device="cuda")
row_max_kernel[(8,)](x, out_good, 100, BLOCK_N=BLOCK_N, BAD_FILL=False)
torch.testing.assert_close(out_good, x.max(dim=1).values)
print("填 -inf:正确 ✓")

out_bad = torch.empty(8, device="cuda")
row_max_kernel[(8,)](x, out_bad, 100, BLOCK_N=BLOCK_N, BAD_FILL=True)
print("填 0.0 :", out_bad.cpu().numpy().round(3), "<- 全错!越界位置的 0 比所有真实负数都大")

### 例 3:一个 program 处理多块 —— 块内循环(grid-stride 思想)

grid 不必恰好覆盖数据:可以只启动固定数量的 program,每个 program 内部**循环**消化多块。当数据特别大、或想精确控制 program 数量(如 persistent kernel)时用这个模式。

In [ ]:
@triton.jit
def add_loop_kernel(x_ptr, y_ptr, out_ptr, n_elements,
                    BLOCK_SIZE: tl.constexpr, NUM_PROGRAMS: tl.constexpr):
    pid = tl.program_id(0)
    # 我处理第 pid, pid+P, pid+2P, ... 块(P = program 总数)
    num_blocks = tl.cdiv(n_elements, BLOCK_SIZE)
    for blk in range(pid, num_blocks, NUM_PROGRAMS):
        offsets = blk * BLOCK_SIZE + tl.arange(0, BLOCK_SIZE)
        mask = offsets < n_elements
        x = tl.load(x_ptr + offsets, mask=mask)
        y = tl.load(y_ptr + offsets, mask=mask)
        tl.store(out_ptr + offsets, x + y, mask=mask)


n = 10_000_000
a, b = torch.randn(n, device="cuda"), torch.randn(n, device="cuda")
out = torch.empty_like(a)
NUM_P = 80  # 只启动 80 个 program(T4 有 40 个 SM,各跑约 2 个)
add_loop_kernel[(NUM_P,)](a, b, out, n, BLOCK_SIZE=1024, NUM_PROGRAMS=NUM_P)
torch.testing.assert_close(out, a + b)
print(f"{n:,} 元素,仅 {NUM_P} 个 program,每个循环 ~{triton.cdiv(n, 1024)//NUM_P} 次:正确 ✓")

---

## §3 知识连接

**与前面篇章:**

- 第 02 篇的 SM:grid 里的每个 program 被调度到某个 SM 上;program 数远多于 SM 数时,SM 跑完一个接着拿下一个——所以例 3 里 80 个 program 也能吃满 40 个 SM;
- 第 03 篇的 add_kernel 是本篇例 1 的一维特例。

**与 CUDA 对照:**

- `grid=(M, N)` ≈ CUDA 的 `dim3 gridDim(M, N)`;`tl.program_id(0/1)` ≈ `blockIdx.x/y`;
- 例 3 的块内循环就是 CUDA 经典的 **grid-stride loop** 在块级的版本;推理框架里的 persistent kernel(常驻 SM 的 kernel,如 vLLM 部分 decode kernel)正是这个模式的极致化。

**与真实框架:**

- vLLM 的 paged attention Triton kernel 用 `grid=(num_seqs, num_heads, ...)` 的多维划分,每个维度的 `program_id` 取不同语义(第几个请求、第几个注意力头)——读懂本篇例 1,那些代码的索引部分就不再可怕;
- `triton.next_power_of_2(N)` + mask 是处理变长行宽的标准 idiom,PyTorch Inductor 生成的归约 kernel 里随处可见。

---

## §4 闭环对比实验:BLOCK_SIZE 与 num_warps 如何影响性能

同一个向量加法,固定数据量 64M 元素,扫 BLOCK_SIZE × num_warps,看性能曲面。`num_warps` 是 kernel 启动参数:一个 program 编译成多少个 warp(默认 4,即 128 线程)。

In [ ]:
import matplotlib.pyplot as plt

@triton.jit
def add_kernel(x_ptr, y_ptr, out_ptr, n_elements, BLOCK_SIZE: tl.constexpr):
    pid = tl.program_id(0)
    offsets = pid * BLOCK_SIZE + tl.arange(0, BLOCK_SIZE)
    mask = offsets < n_elements
    tl.store(out_ptr + offsets,
             tl.load(x_ptr + offsets, mask=mask) + tl.load(y_ptr + offsets, mask=mask),
             mask=mask)


n = 64 * 1024 * 1024
a, b = torch.randn(n, device="cuda"), torch.randn(n, device="cuda")
out = torch.empty_like(a)
bytes_moved = 3 * n * 4

block_sizes = [64, 128, 256, 512, 1024, 2048, 4096, 8192]
warp_options = [1, 2, 4, 8]

results = {}
print(f"{'BLOCK_SIZE':>10} | " + " | ".join(f"warps={w:>2}" for w in warp_options) + "   (GB/s)")
for bs in block_sizes:
    row = []
    for w in warp_options:
        grid = (triton.cdiv(n, bs),)
        ms = gpu_time_ms(lambda: add_kernel[grid](a, b, out, n, BLOCK_SIZE=bs, num_warps=w))
        row.append(bytes_moved / (ms / 1000) / 1e9)
    results[bs] = row
    print(f"{bs:>10} | " + " | ".join(f"{v:>8.1f}" for v in row))

plt.figure(figsize=(9, 4.5))
for i, w in enumerate(warp_options):
    plt.semilogx(block_sizes, [results[bs][i] for bs in block_sizes], "o-", label=f"num_warps={w}")
plt.axhline(320, color="gray", ls="--", label="T4 theoretical 320 GB/s")
plt.xlabel("BLOCK_SIZE")
plt.ylabel("effective bandwidth (GB/s)")
plt.title("Vector add: BLOCK_SIZE x num_warps sweep (64M elements)")
plt.legend(); plt.grid(True, alpha=0.3)
plt.show()

best = max(((bs, w) for bs in block_sizes for w in warp_options),
           key=lambda p: results[p[0]][warp_options.index(p[1])])
print(f"最优组合: BLOCK_SIZE={best[0]}, num_warps={best[1]}")

### 实验结果解读

- **太小的 BLOCK_SIZE**(64/128)性能差:program 数量爆炸,启动调度开销大,且每个 warp 的访存请求太零碎,无法充分流水;
- **中间档**(512~4096 配 4~8 warps)进入平台期:带宽已打满,继续调参没有收益——memory-bound 算子的特征;
- **BLOCK_SIZE 与 num_warps 要匹配**:块大 warp 少,意味着每个线程处理很多元素(寄存器压力上升);块小 warp 多则线程没活干。没有普适最优解——**这正是第 11 篇 `@triton.autotune` 存在的理由:让机器替你扫这个表**;
- 结论先记下:逐元素类 kernel 对参数不敏感(平台很宽),GEMM 类 kernel 极其敏感(第 11 篇可见差几倍)。

---

## §5 练习 + 面试考点

### 动手练习

1. 把例 1 改成"每个 program 处理一整行"(`grid=(M,)`,行内用 `tl.arange(0, BLOCK_N)` + mask 覆盖 777 列),与二维分块版本比性能。想一想:什么形状的矩阵下两种划分差距最大?
2. 用例 2 的框架实现 `row_sum`(行求和),故意把 `other` 填成 1.0,观察误差随"越界长度"(BLOCK_N - N)的变化。

### 面试高频考点

- **Q:Triton 里两个 program 能同步/通信吗?**
  A:不能。Triton 没有跨 program 的 barrier;grid 内的 program 执行顺序无保证。需要全局协作时,要么拆成两个 kernel(kernel 边界天然是全局同步点),要么用 atomic 操作做无序累加(第 13 篇),要么用 persistent kernel + 自旋锁等高级技巧(超出 Triton 舒适区)。
- **Q:数据长度不是 2 的幂、也不被 BLOCK 整除,Triton 怎么处理?**
  A:grid 用 `cdiv` 向上取整,块内用 `mask` 截断;归约场景还要给 `tl.load` 配正确的 `other` 填充值(sum 填 0,max 填 -inf)。
- **Q:BLOCK_SIZE 怎么选?**
  A:没有静态答案——它影响寄存器用量、shared memory、occupancy 和访存效率,与算子类型和 GPU 型号都相关。工程做法是 `@triton.autotune` 给一组候选配置自动实测挑选;经验上逐元素算子 512~2048 都行,GEMM 类需要认真调。
- **Q:一个 Triton program 对应硬件上的什么?**
  A:一个线程块(CTA),被调度到一个 SM 上;`num_warps` 决定它包含多少 warp。program 数量应远大于 SM 数以隐藏延迟(或用 persistent 模式精确等于可常驻数)。